In [263]:
import pandas as pd
import os
import geopandas as gpd

In [264]:
directory = 'new_sites_timeslice'

# Check if the directory exists, and create it if it doesn't
if not os.path.exists(directory):
    os.makedirs(directory)
    print(f"Directory '{directory}' created successfully.")
else:
    print(f"Directory '{directory}' already exists.")


Directory 'new_sites_timeslice' already exists.


In [265]:
season_config = {
    "Winter1": {"start": "2021-01-01", "end": "2021-03-19", "daytime": {"day": 8, "night": 17}},
    "Spring": {"start": "2021-03-20", "end": "2021-06-19", "daytime": {"day": 6, "night": 20}},
    "Summer": {"start": "2021-06-20", "end": "2021-09-21", "daytime": {"day": 6, "night": 21}},
    "Fall": {"start": "2021-09-22", "end": "2021-12-20", "daytime": {"day": 8, "night": 18}},
    "Winter2": {"start": "2021-12-21", "end": "2021-12-31", "daytime": {"day": 8, "night": 17}}
}

# Mapping dictionary
timeslice_name_mapping = {
    'Fall_D': 'SEA2D',
    'Fall_N': 'SEA2N',
    'Out_of_Season': 'NA',
    'Spring_D': 'SEA3D',
    'Spring_N': 'SEA3N',
    'Summer_D': 'SEA4D',
    'Summer_N': 'SEA4N',
    'Winter1_D': 'SEA1D',
    'Winter1_N': 'SEA1N',
    'Winter2_D': 'SEA1D',
    'Winter2_N': 'SEA1N'
}


In [266]:
def create_timeslice_from_cluster(cluster_ts_pickle,site_name,season_config,timeslice_name_mapping):

    df=pd.read_pickle(cluster_ts_pickle)

    # Convert season_config dates to datetime with error handling
    for season, dates in season_config.items():
        try:
            dates["start"] = pd.to_datetime(dates["start"])
            dates["end"] = pd.to_datetime(dates["end"])
        except ValueError as e:
            print(f"Error converting dates for season '{season}': {e}")

    # Improved function for clarity and error handling
    def get_season_time_of_day(datetime):
        for season, dates in season_config.items():
            if dates["start"] <= datetime <= dates["end"]:
                # Handle specific time ranges for day and night
                if 0 <= datetime.hour <= dates["daytime"]["day"]:
                    return f"{season}_N"  # Night
                elif dates["daytime"]["day"] <= datetime.hour <= dates["daytime"]["night"]:
                    return f"{season}_D"  # Day
                else:  # Outside daytime range
                    return f"{season}_N"  # Assuming night for values outside day/night times
        # Handle date outside any season (optional: return specific value or raise an error)
        return "Out_of_Season"  # Example handling

    # Create the 'season_daytime' column with error handling
    try:
        df['season_daytime'] = df.index.map(get_season_time_of_day)
    except TypeError as e:
        print("Error applying function: ", e)

    # Display the DataFrame
    # Group by season and day/night
    grouped_df = df.groupby(['season_daytime']).agg({
        site_name: ['mean', 'std']  # You can add more aggregation functions here
    })

    # Display the grouped DataFrame
    grouped_df['CF_mean'] = grouped_df[site_name]['mean']
    grouped_df['timeslices'] = grouped_df.index.map(timeslice_name_mapping)
    grouped_df=grouped_df.groupby('timeslices').mean()
    return grouped_df

In [267]:
def create_n_save_timeslice_files(ts_pickle,cluster_pickle,asset_type,season_config,timeslice_name_mapping):
    sites_df=gpd.GeoDataFrame(pd.read_pickle(cluster_pickle))
    
    sites=list(sites_df.index)
    cluster_ts_timeslice = {}

    for site in sites:
        cluster_ts_timeslice[site] = create_timeslice_from_cluster(ts_pickle, site,season_config,timeslice_name_mapping)
        cluster_ts_timeslice[site].to_csv(f'new_sites_timeslice/{asset_type}_{site}.csv')
    print(f"BCNexus compatible timeseries created for - {asset_type} Clusters")
    return cluster_ts_timeslice

In [268]:
solar_timeslices=create_n_save_timeslice_files('Solar_Top_Sites_Clustered_CF_timeseries.pkl','Solar_Top_Sites_Clustered.pkl','Solar',season_config,timeslice_name_mapping)
windd_timeslices=create_n_save_timeslice_files('Wind_Top_Sites_Clustered_CF_timeseries.pkl','Wind_Top_Sites_Clustered.pkl','Wind',season_config,timeslice_name_mapping)

BCNexus compatible timeseries created for - Solar Clusters
BCNexus compatible timeseries created for - Wind Clusters


In [271]:
solar_timeslices['East Kootenay_1']

East Kootenay_1             CF_mean
                      mean       std          
timeslices                                    
NA                0.239899  0.308992  0.239899
SEA1D             0.046312  0.130219  0.046312
SEA1N             0.207803  0.278994  0.207803
SEA2D             0.103268  0.197755  0.103268
SEA2N             0.188932  0.265236  0.188932
SEA3D             0.305800  0.334536  0.305800
SEA3N             0.347606  0.300090  0.347606
SEA4D             0.318369  0.317546  0.318369
SEA4N             0.299507  0.263545  0.299507